# Modded nanoGPT speedrun: sparse basis embeddings

Standard token tables treat every type as an independent vector. A batch that contains `cat` updates `cat` and nothing else.

Here each token is a **linear combination of a shared basis**:

$$e_i = c_i^\top B$$

with optional TopK / ReLU sparsity on the codes $c_i$. Then a gradient into a basis atom moves **every token that uses that atom**, including types that did not appear in the batch. That is the sample-efficiency claim, not just a rare-word quality patch.

This is a CPU-scale cousin of the [modded-nanoGPT](https://github.com/KellerJordan/modded-nanogpt) speedrun: same game (tokens and wall time to a val-loss target), tiny WikiText-2 GPT, three embedding parameterizations.

## Variants

| name | embedding |
|---|---|
| `dense` | $e_i$ is a lookup row (nanoGPT default, tied to the LM head) |
| `factorized` | dense $c_i \in \mathbb{R}^{k}$, $k < d$ (ALBERT-style) |
| `sparse` | ReLU + TopK codes over $k$ atoms (this experiment) |

The transformer is otherwise the same modded nanoGPT block: RMSNorm, RoPE, QK-norm, ReLU² MLP, Muon on hidden matrices, AdamW on embeddings.

## Run

```bash
uv run python -m pytorch_katas.nanogpt.train --smoke          # ~10s sanity
uv run python -m pytorch_katas.nanogpt.train --embedding all  # full CPU speedrun
```

Logs land in `data/nanogpt/logs/` (gitignored). Curves and a copy of the log sit next to this notebook.

## CPU speedrun result (WikiText-2, 1.23M tokens, 4-layer 128-d GPT)

| embedding | params | min val | final val | rare/freq L2 drift |
|---|---|---|---|---|
| dense | 4.44M | 5.13 | 6.25 (diverged after 300k tokens) | 3.24 |
| factorized | 2.62M | **5.02** | **5.02** | 5.51 |
| sparse TopK | 2.62M | 5.32 | 5.35 | 0.25 |

Sharing the basis **did** buy sample efficiency versus a dense table: both factorized and sparse avoid the dense-run collapse and finish lower. The dense ALBERT-style factorisation won. Random TopK support is a weak sharing graph — rare tokens keep their initial 8 random atoms, which often are not the atoms that training actually updates, so they barely move. Factorized codes use every atom, so a rare type rides the whole basis.

In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display

from pytorch_katas.settings import DATA_DIR

nb_dir = Path.cwd() if Path.cwd().name == "nanogpt" else Path("notebooks/nanogpt")
plot = nb_dir / "speedrun_curves.png"
log_path = DATA_DIR / "nanogpt" / "logs" / "speedrun.json"
if not log_path.exists():
    log_path = nb_dir / "speedrun.json"

if plot.exists():
    display(Image(filename=str(plot)))
if log_path.exists():
    payload = json.loads(log_path.read_text())
    for result in payload["results"]:
        reached = result["reached_target"]
        print(
            f"{result['embedding_type']:12}  params={result['params']:,}  "
            f"final_val={result['final_val']:.4f}  min_val={result.get('min_val', result['final_val']):.4f}  "
            f"rare/freq drift={result['drift']['rare_to_frequent']:.3f}  "
            f"tokens_to_target={reached['tokens']}"
        )

## What to look at

1. **Val loss vs tokens** — did sharing buy sample efficiency, or just compress the table?
2. **Rare vs frequent embedding drift** — in `dense`, rare types barely move. In `sparse` / `factorized`, they should ride the shared basis even when they are not in the batch.

The unit test `test_shared_basis_moves_unused_token` is the mechanistic version of (2): token 4 never appears in the forward pass, but its embedding still changes because it shares atom 1 with token 0.